# ¿Cómo entrenamos el modelo de emociones?

In [15]:
import cv2
from roboflow import Roboflow

rf = Roboflow(api_key="klmVEuTiYxKVkKRwtyGF")
project = rf.workspace("emotions-dectection").project("human-face-emotions")
version = project.version(30)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
from ultralytics import YOLO

# El modelo se cargará en la GPU si está disponible
model = YOLO("yolo11n.pt") 
data_path = "./Human-face-emotions-30/data.yaml"

# El entrenamiento usará la GPU automáticamente
results = model.train(data=data_path, epochs=50, imgsz=640)

# Prueba de detección de emociones

In [ ]:
import cv2
from ultralytics import YOLO

EMOTICONS = {}
def load_icon(path, force_bgr=False):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"Advertencia: No se pudo cargar el archivo: {path}")
        return None
    if force_bgr and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img

# 1. Cargar las imágenes
emoticono_feliz = load_icon('feliz.png')
emoticono_triste = load_icon('triste.png')
emoticono_enfadado = load_icon('enfadado.png')
emoticono_asqueroso = load_icon('asqueroso.png')
emoticono_miedo = load_icon('miedo.png')
emoticono_neutral = load_icon('neutral.jpg', force_bgr=True) 
emoticono_sorpresa = load_icon('sorpresa3.png',force_bgr=True) 
    
# 2. Crear el diccionario solo con los que se cargaron
if emoticono_feliz is not None:
    EMOTICONS["happy"] = emoticono_feliz
    EMOTICONS["content"] = emoticono_feliz
if emoticono_triste is not None:
    EMOTICONS["sad"] = emoticono_triste
if emoticono_enfadado is not None:
    EMOTICONS["anger"] = emoticono_enfadado
if emoticono_asqueroso is not None:
    EMOTICONS["disgust"] = emoticono_asqueroso
if emoticono_miedo is not None:
    EMOTICONS["fear"] = emoticono_miedo
if emoticono_neutral is not None:
    EMOTICONS["neutral"] = emoticono_neutral
if emoticono_sorpresa is not None:
    EMOTICONS["surprise"] = emoticono_sorpresa



color_emotions = {
    "happy": (0, 255, 0), "sad": (255, 0, 0), "anger": (0, 0, 255),
    "surprise": (255, 255, 0), "neutral": (255, 165, 0), "content": (0, 150, 0),
    "fear": (0, 69, 255), "disgust": (128, 0, 128), "background": (255, 255, 255),
} 

capture_video = cv2.VideoCapture(0)
model = YOLO("runs/detect/train2/weights/best.pt")

last_box_coords = None

while True:
    ret, frame = capture_video.read()
    if not ret:
        break
        
    results = model(frame, conf=0.5, verbose=False)
    current_emotion = None 
    last_box_coords = None 
    
    for result in results:
        boxes = result.boxes
        if len(boxes) == 0:
            continue
            
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0].item()
            class_id = int(box.cls[0].item())
            label = model.names.get(class_id, "unknown")
            
            last_box_coords = (x1, y1, x2, y2)
            current_emotion = label
            
            color_emotion = color_emotions.get(label, (255, 255, 255))
            
            if label not in ["background", "unknown"]:
                print( "Label:", label )
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), color_emotion, 2)
            cv2.putText(frame, f"{label} {confidence:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color_emotion, 2)
            
            break 
    
    
    if last_box_coords and current_emotion in EMOTICONS:
        x1, y1, x2, y2 = last_box_coords
        
        emotion_icon = EMOTICONS[current_emotion]
        
        face_width = x2 - x1
        ICON_SIZE = int(face_width * 0.5) 
        
        if emotion_icon is not None and ICON_SIZE > 0:
            
            icon_resized = cv2.resize(emotion_icon, (ICON_SIZE, ICON_SIZE), interpolation=cv2.INTER_AREA)
            
            y_start_calc = y1 - ICON_SIZE
            x_start_calc = x1 + int((face_width / 2) - (ICON_SIZE / 2)) 
            y_offset = max(0, -y_start_calc) 
            x_offset = max(0, -x_start_calc) 
            y_start = max(0, y_start_calc)  
            x_start = max(0, x_start_calc)  
            y_end = min(frame.shape[0], y_start_calc + ICON_SIZE) 
            x_end = min(frame.shape[1], x_start_calc + ICON_SIZE) 
            roi = frame[y_start:y_end, x_start:x_end]
            icon_to_paste = icon_resized[y_offset : y_offset + roi.shape[0], 
                                         x_offset : x_offset + roi.shape[1]]
            if icon_to_paste.shape[2] == 4:
                b, g, r, a = cv2.split(icon_to_paste)
                alpha = a / 255.0
                inv_alpha = 1.0 - alpha
                icon_bgr = cv2.merge([b, g, r])
                for c in range(0, 3):
                    roi[:, :, c] = (alpha * icon_bgr[:, :, c]) + \
                                   (inv_alpha * roi[:, :, c])
            else:
                roi[:, :] = icon_to_paste[:, :]
                
    # 6. Mostrar el frame
    cv2.imshow("Emotion Detection - Live", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 7. Liberar recursos
capture_video.release()
cv2.destroyAllWindows()